# Notebook 5.3: Simplified Droplet Advection (Optional)

## Objective
Model transport *inside* a droplet moving through a microfluidic channel.

**Simplifications:**
- The droplet moves as a rigid body (no deformation)
- External Poiseuille flow sets the droplet velocity = $U_{drop}$
- Internal flow = Hill vortex (analytical recirculation)
- Solve convection-diffusion inside the moving droplet domain

**This models:** mixing inside a droplet due to internal recirculation — key for droplet-based assays.

In [ ]:
from fenics import *
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
set_log_level(LogLevel.WARNING)

---
## Step 1: Droplet Domain

We model a 2D circular droplet with radius $R$, meshed as a disc.

In [ ]:
import gmsh
import meshio

R   = 0.04   # Droplet radius (m)
lc  = 0.004  # Mesh size

gmsh.initialize()
gmsh.model.add("droplet")

# Create disc
gmsh.model.geo.addPoint(0, 0, 0, lc, 1)          # centre
gmsh.model.geo.addPoint(R, 0, 0, lc, 2)           # right
gmsh.model.geo.addPoint(0, R, 0, lc, 3)           # top
gmsh.model.geo.addPoint(-R, 0, 0, lc, 4)          # left
gmsh.model.geo.addPoint(0, -R, 0, lc, 5)          # bottom

gmsh.model.geo.addCircleArc(2, 1, 3, 1)
gmsh.model.geo.addCircleArc(3, 1, 4, 2)
gmsh.model.geo.addCircleArc(4, 1, 5, 3)
gmsh.model.geo.addCircleArc(5, 1, 2, 4)

gmsh.model.geo.addCurveLoop([1, 2, 3, 4], 1)
gmsh.model.geo.addPlaneSurface([1], 1)

gmsh.model.addPhysicalGroup(1, [1,2,3,4], name="droplet_surface")
gmsh.model.addPhysicalGroup(2, [1], name="droplet")

gmsh.model.geo.synchronize()
gmsh.model.mesh.generate(2)
gmsh.model.mesh.setOrder(2)
gmsh.write("/tmp/droplet.msh")
gmsh.finalize()

# Convert to XML
msh = meshio.read("/tmp/droplet.msh")
cells = {"triangle": msh.cells_dict["triangle"]}
meshio.write("/tmp/droplet.xml",
             meshio.Mesh(points=msh.points[:, :2], cells=cells))

mesh = Mesh("/tmp/droplet.xml")
print(f"Droplet mesh: {mesh.num_cells()} cells")

---
## Step 2: Analytical Internal Velocity (Hill Vortex)

For a droplet in Stokes flow, the internal velocity field (in the droplet frame) is approximately:
$$u_x = A\, y(R^2 - x^2 - y^2), \quad u_y = -A\, x(R^2 - x^2 - y^2)$$

This creates a toroidal recirculation.

In [ ]:
A = 5.0  # Recirculation strength

# Express as FEniCS Expression
u_internal = Expression(
    ("A * x[1] * (R*R - x[0]*x[0] - x[1]*x[1])",
     "-A * x[0] * (R*R - x[0]*x[0] - x[1]*x[1])"),
    degree=3, A=A, R=R
)

V_vec = VectorFunctionSpace(mesh, "P", 2)
u_proj = project(u_internal, V_vec)

# Visualise internal flow
S_plot = FunctionSpace(mesh, "P", 1)
u_mag_plot = project(sqrt(u_proj[0]**2 + u_proj[1]**2), S_plot)

fig, ax = plt.subplots(figsize=(6, 6))
c = plot(u_mag_plot, ax=ax, cmap='viridis')
plt.colorbar(c, ax=ax, label='$|u|$')
ax.set_title('Hill vortex — internal droplet flow', fontsize=12)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('/tmp/droplet_flow.png', dpi=150)
plt.show()

---
## Step 3: Convection-Diffusion Inside Droplet

In [ ]:
D   = 0.001
dt  = 0.02
T   = 2.0

S   = FunctionSpace(mesh, "P", 1)
c   = TrialFunction(S)
phi = TestFunction(S)
c_n = interpolate(Expression("x[0] > 0 ? 1.0 : 0.0", degree=1), S)  # Left half=0, right half=1

a_c = (c/dt * phi + dot(u_proj, grad(c))*phi + D*dot(grad(c), grad(phi))) * dx
L_c = c_n/dt * phi * dx

# No BC on droplet surface (sealed droplet)
c_sol = Function(S)
t = 0
snapshots = {}

while t < T - 1e-8:
    t += dt
    b_c = assemble(L_c)
    A_c = assemble(a_c)
    solve(A_c, c_sol.vector(), b_c)
    c_n.assign(c_sol)
    for ts in [0.2, 0.5, 1.0, 2.0]:
        if abs(t - ts) < dt/2 and ts not in snapshots:
            snapshots[ts] = c_sol.copy(deepcopy=True)
            std = c_sol.vector().get_local().std()
            print(f"  t = {ts:.1f}, std(c) = {std:.4f}  ({'well mixed' if std < 0.05 else 'mixing'})")

print("Done.")

---
## Step 4: Visualise Mixing Progress

In [ ]:
n = len(snapshots)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
if n == 1: axes = [axes]

for ax, (ts, cs) in zip(axes, sorted(snapshots.items())):
    im = plot(cs, ax=ax, cmap='RdBu_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_title(f't = {ts}', fontsize=11)
    ax.set_aspect('equal')

plt.suptitle('Mixing inside droplet (Hill vortex)', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/droplet_mixing.png', dpi=150)
plt.show()

---
## Summary

This simplified model demonstrates:
- The **Hill vortex** promotes internal mixing via recirculation
- Even without external convection, the internal flow homogenises concentration
- Mixing time scales as $t_{mix} \sim R^2 / D$ (diffusion) but is reduced by $A$ (recirculation)

**For full droplet microfluidics simulation** you would need:
- Interface tracking (level set / phase field)
- Two-phase Navier-Stokes
- Surfactant transport

This is a large research problem. This simplified model is suitable for screening designs.

**Exercise:** Vary `A` (recirculation strength). How does mixing time scale with `A`?